In [ ]:
from sampo.hybrid.population import PopulationScheduler
import numpy as np
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import sampo.scheduler

from sampo.userinput.parser.csv_parser import CSVParser
from sampo.hybrid.cycle import CycleHybridScheduler
from sampo.api.genetic_api import ScheduleGenerationScheme, Individual
from sampo.scheduler import HEFTScheduler, HEFTBetweenScheduler, TopologicalScheduler, GeneticScheduler
from sampo.hybrid.population import HeuristicPopulationScheduler, GeneticPopulationScheduler

from sampo.generator.environment import get_contractor_by_wg
from sampo.generator import SimpleSynthetic

from sampo.base import SAMPO
from sampo.schemas import WorkGraph
from sampo.schemas.time import Time

# Импорт ресурсной модели
from models.field_dev_resources_time_estimator import FieldDevWorkEstimator

# Фитнесс-функции
from sampo.scheduler.genetic.operators import TimeFitness, DeadlineCostFitness

# Потом можно заменить, если добавить в функциях аргументы по умолчанию
from sampo.schemas.landscape import LandscapeConfiguration
from sampo.schemas.schedule_spec import ScheduleSpec

# ??
from my_utils.precedence import PrecedenceManager
from my_utils.project_info import ProjectInfo

# Визуализация
import plotly.express as px
import plotly.graph_objects as go

## Get project data

In [ ]:
DATA_PATH = Path('data/csv/gas_network_full_connections.csv')
HISTORY_PATH = Path('data/historical_projects_data.csv')

project_df = pd.read_csv(DATA_PATH, sep=',')
history_df = pd.read_csv(HISTORY_PATH, sep=';')

all_connections = True
change_connections = False

project_work_estimator = FieldDevWorkEstimator()

In [ ]:
%%capture
# not print output

wg, contractors = CSVParser.work_graph_and_contractors(
    works_info=CSVParser.read_graph_info(
        project_info=project_df,
        history_data=history_df,
        all_connections=all_connections,
        change_connections_info=change_connections
    ),
    work_resource_estimator=project_work_estimator
)

## My scheduler

In [ ]:
project_info = ProjectInfo(wg, contractors)
precedence_manager = PrecedenceManager(project_info.children, project_info.parents)

In [ ]:
class BeeScheduler(PopulationScheduler):
    def __init__(self):
        pass

    def schedule(self,
                 initial_population: list,
                 wg: WorkGraph,
                 contractors: list,
                 spec: ScheduleSpec = ScheduleSpec(),
                 assigned_parent_time: Time = Time(0),
                 landscape: LandscapeConfiguration = LandscapeConfiguration()) -> list:

        colony = BeeColony.from_population(initial_population)
        for i in range(1):
            colony.mutate()
        population = colony.to_population()
        return population

In [ ]:
class Bee:

    def __init__(self, id_, activity_list, resources_matrix, ceil_list, fitness=None, n_tries=0):
        self.id_ = id_
        self.activity_list = np.array(activity_list).copy()
        self.resources_matrix = resources_matrix.copy()
        self.ceil_list = ceil_list.copy()
        self.fitness = fitness
        self.n_tries = n_tries

    @classmethod
    def from_individual(cls, id_, individual):
        return cls(
            id_=id_,
            activity_list=individual[0], 
            resources_matrix=individual[1], 
            ceil_list=individual[2], 
            fitness=individual.fitness.values[0]
        )
    
    def reset(self):
        return Bee(
            id_=self.id_,
            activity_list=np.array(get_random_chromosome()[0]),
            resources_matrix=get_random_chromosome()[1],
            ceil_list=get_random_chromosome()[2],
            fitness=None,
            n_tries=0
        )

    def random_mutation(self):

        new_activity_list = self.activity_list.copy()
        new_resources_matrix = self.resources_matrix.copy()
        
        n_resource_mutations = np.random.randint(0, 5+1)
        for _ in range(n_resource_mutations):
            work_to_mutate, worker_to_mutate = project_info.nonzero_resources_indices[np.random.choice(len(project_info.nonzero_resources_indices))]
            min_amount, max_amount = project_info.get_resource_borders(work_id=work_to_mutate, worker_id=worker_to_mutate)
            new_resources_matrix[work_to_mutate, worker_to_mutate] = np.random.randint(min_amount, max_amount+1)
    
        n_order_mutations = np.random.randint(0, 5+1)
        for _ in range(n_order_mutations):
            a, b = np.random.randint(project_info.n_works, size=2)
            new_activity_list[[a, b]] = new_activity_list[[b, a]]
        new_activity_list = precedence_manager.convert_al_to_valid_order(new_activity_list)

        return Bee(self.id_, new_activity_list, new_resources_matrix, self.ceil_list.copy(), fitness=None, n_tries=0)

    def towards_mutation(self, other):

        # TODO how to do a "little" crossover??
        new_activity_list = self.activity_list.copy()
        # new_activity_list = np.where(
        #     np.random.choice([0, 1], p=[0.90, 0.10], size=project_info.n_works),
        #     self.activity_list,
        #     other.activity_list
        # )
        
        new_resources_matrix = np.where(
            np.random.choice([0, 1], p=[0.90, 0.10], size=(project_info.n_works, project_info.n_workers+1)),
            self.resources_matrix,
            other.resources_matrix
        )

        return Bee(self.id_, new_activity_list, new_resources_matrix, self.ceil_list.copy(), fitness=None, n_tries=0)

In [ ]:
class BeeColony:

    def __init__(self, bees):
        self.bees = bees
        self.n_bees = len(bees)

    @classmethod
    def from_population(cls, population):
        return cls([Bee.from_individual(id_, i) for id_, i in enumerate(population)])

    def to_population(self):
        individuals = [
            Individual(
                individual_fitness_constructor=TimeFitness,
                chromosome=[bee.activity_list, bee.resources_matrix, bee.ceil_list, ScheduleSpec(), np.array([])]
            )
            for bee in self.bees
        ]
        for i in range(len(individuals)):
            individuals[i].fitness.values = (self.bees[i].fitness, )

        return individuals
    
    def mutate(self):
        # get new bees
        new_bees = []

        # random direction bees
        for _ in range(self.n_bees):
            bee_index = np.random.choice(self.n_bees)
            new_bees.append( self.bees[bee_index].random_mutation() )
        
        # toward other bees
        for _ in range(self.n_bees):
            bee_1_index = np.random.choice(self.n_bees)
            bee_2_index = np.random.choice(self.n_bees)
            new_bees.append(
                self.bees[bee_1_index].towards_mutation(self.bees[bee_2_index])
            )
        
        # calculate fitness
        new_fitness = current_single_fitness_function(new_bees)
        for i in range(len(new_bees)):
            new_bees[i].fitness = new_fitness[i]

        # update positions
        for i in range(len(new_bees)):
            # if improve: move
            if new_bees[i].fitness < self.bees[new_bees[i].id_].fitness:
                self.bees[new_bees[i].id_] = new_bees[i]
            # if no improve: remember it
            # else:
            #     self.bees[new_bees[i].id_].n_tries += 1

        # scout phase
        # scout_index = []
        # for i in range(len(self.bees)):
        #     if self.bees[i].n_tries >= 20:
        #         scout_index.append(i)
        #         self.bees[i] = self.bees[i].reset()
        # fitness = current_single_fitness_function([self.bees[i] for i in scout_index])
        # for i in range(len(scout_index)):
        #     self.bees[scout_index[i]].fitness = fitness[i]
        

    def get_fitness_stats(self):
        fitness_array = np.array([bee.fitness for bee in self.bees])
        return np.mean(fitness_array), np.min(fitness_array)

In [ ]:
fitness_object = DeadlineCostFitness(deadline=Time(400))

In [ ]:
def current_single_fitness_function(bees):
    fitness_array = SAMPO.backend.compute_chromosomes(fitness_object, [
        [bee.activity_list, bee.resources_matrix, bee.ceil_list, ScheduleSpec(), np.array([])]
        for bee in bees
    ])
    return [i[0] for i in fitness_array]

## Test

In [ ]:
heuristics = HeuristicPopulationScheduler([
    HEFTScheduler(work_estimator=project_work_estimator), 
    HEFTBetweenScheduler(work_estimator=project_work_estimator), 
    TopologicalScheduler(work_estimator=project_work_estimator)
])

genetic_1 = GeneticPopulationScheduler(GeneticScheduler(number_of_generation=1,
                                                        size_of_population=50,
                                                        mutate_order=0.01,
                                                        mutate_resources=0.01,
                                                        sgs_type=ScheduleGenerationScheme.Parallel,
                                                        work_estimator=project_work_estimator,
                                                        fitness_constructor=fitness_object))

bees = BeeScheduler()

hybrid_combine = CycleHybridScheduler(heuristics, [genetic_1], max_plateau_size=10, max_steps=100, fitness=fitness_object)

# wg = WorkGraph.load('wgs', f'{graph_size}_{iteration}')
# contractors = [get_contractor_by_wg(wg)]

# SAMPO.backend = MultiprocessingComputationalBackend(n_cpus=10)
SAMPO.backend.cache_scheduler_info(wg, contractors, work_estimator=project_work_estimator)
SAMPO.backend.cache_genetic_info()

pop = hybrid_combine.run(wg, contractors)

In [ ]:
history = hybrid_combine.history

In [ ]:
len(history)

In [ ]:
gen_mins_g = [np.min([i[0] for i in a[1]]) for a in history]
# gen_mins_g_b = [np.min([i[0] for i in a[1]]) for a in history]

In [ ]:
fig = go.Figure([
    go.Scatter(y=gen_mins_g, name="Генетика", line_color="green"),
    go.Scatter(y=gen_mins_g_b[:201:2], name="Генетика + Пчелы", line_color="navy"),    
])

fig.update_layout(height=500, width=1000, template="plotly_white")
fig.update_layout(
    xaxis_title="Итерация",                   
    yaxis_title="Метрика (стоимость + штраф)",
    showlegend=True,
    margin_t=0, margin_b=0, margin_l=0, margin_r=0
)

fig.update_layout(legend=dict(
    yanchor="top", y=0.95,
    xanchor="right", x=0.95
))

fig.show()
fig.write_image('plots/first_hybrid_result_3.png', scale=4)

In [ ]:
355.9/344.1